# **Import**


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.tree import DecisionTreeRegressor
from data_loader import Dataset
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

# **Build From Scratch**


In [2]:
class CustomGradientBoosting():
    
    def __init__(self, n_estimators=300, learning_rate=0.1, max_depth=16):

        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

    def _compute_F0(self, y):

        if self.problem_type == "binary classification":
            return np.log(sum(y == 1) / sum(y == 0))
        else:
            return np.mean(y)

    def _get_wl_target(self, y, F_h):

        if self.problem_type == "binary classification":
            y_pred = 1 / (1 + np.exp(-F_h))

        else:
            y_pred = F_h

        return y - y_pred

    def fit(self, X, y):

        if len(set(y)) == 2:
            self.problem_type = "binary classification"
        else:
            self.problem_type = "regression"

        self.wls = []
        self.F_0 = self._compute_F0(y)

        F_h = self.F_0

        for _ in range(self.n_estimators):

            wl = DecisionTreeRegressor(max_depth=self.max_depth)

            wl_target = self._get_wl_target(y, F_h)

            wl.fit(X, wl_target)

            F_h += self.learning_rate * wl.predict(X)

            self.wls.append(wl)

    def predict(self, X):

        F_k = self.F_0 + self.learning_rate * np.sum([wl.predict(X) for wl in self.wls], axis=0)

        if self.problem_type == "binary classification":

            y_pred = 1 / (1 + np.exp(-F_k))

            y_pred = np.array([1 if i >= 0.5 else 0 for i in y_pred])

        else:
            y_pred = F_k

        return y_pred

# **Load and Split**


In [3]:
dataset_b = Dataset("binary classification")
X_train_b, X_test_b, y_train_b, y_test_b = dataset_b.load_split_data()

dataset_r = Dataset("regression")
X_train_r, X_test_r, y_train_r, y_test_r = dataset_r.load_split_data()

# **Train, Test and Compare**



In [4]:
custom_model_b = CustomGradientBoosting()
custom_model_b.fit(X_train_b, y_train_b)
y_pred_custom_b = custom_model_b.predict(X_test_b)
print(f"Custom Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_custom_b):.3f}")

custom_model_r = CustomGradientBoosting()
custom_model_r.fit(X_train_r, y_train_r)
y_pred_custom_r = custom_model_r.predict(X_test_r)
print(f"Custom Regression MSE: {mean_squared_error(y_test_r, y_pred_custom_r):.3f}")

Custom Binary Classification Accuracy: 0.949
Custom Regression MSE: 0.290


In [5]:
sklearn_model_b = GradientBoostingClassifier()
sklearn_model_b.fit(X_train_b, y_train_b)
y_pred_sklearn_b = sklearn_model_b.predict(X_test_b)
print(f"Scikit-learn Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_sklearn_b):.3f}")

sklearn_model_r = GradientBoostingRegressor()
sklearn_model_r.fit(X_train_r, y_train_r)
y_pred_sklearn_r = sklearn_model_r.predict(X_test_r)
print(f"Scikit-learn Regression MSE: {mean_squared_error(y_test_r, y_pred_sklearn_r):.3f}")

Scikit-learn Binary Classification Accuracy: 0.993
Scikit-learn Regression MSE: 0.298
